# WSI Atlas — Batch X-Ray Scanner (Colab Free T4 / L4 / A100)

Escanea automáticamente modelos HuggingFace (`safetensors`) y `.gguf` con **`wsi.xray`** + **`wsi.gguf_loader`**:
- **Auto-resume en Google Drive** (`/content/drive/MyDrive/wsi_xray_atlas/`): si se corta Colab, retoma exactamente donde quedó.
- **Auto-limpieza de disco**: borra el caché de HuggingFace después de cada modelo para no llenar el disco de Colab.
- **Auto-detección de GPU**: usa `fp16` en T4 (16 GB) y `bf16` en L4/A100, saltando automáticamente modelos que no caben en VRAM.


In [ ]:
# @title 1. Setup repo, Drive y dependencias
import os, shutil, subprocess, torch
from pathlib import Path

USE_DRIVE = True  # @param {type:"boolean"}
BRANCH = "exp/xray-atlas"  # @param {type:"string"}

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/wsi_xray_atlas")
else:
    OUT_DIR = Path("/content/wsi_xray_atlas")
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/WEIGHT-SPACE-INDEX")
if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "-b", BRANCH,
        "https://github.com/krexito/weight-space-index.git", str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

subprocess.run([
    "pip", "install", "-q", "-U",
    "transformers==5.12.1", "accelerate==1.15.0", "safetensors==0.8.0",
    "gguf>=0.14.0", "huggingface_hub>=0.28.0", "scipy>=1.11.0"
], check=True)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9) if torch.cuda.is_available() else 0.0
DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
print(f"GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | dtype: {DTYPE} | OUT_DIR: {OUT_DIR}")


In [ ]:
# @title 2. Cola de modelos a capturar (edita o agrega los que quieras)
# Cada entrada: dict(repo=..., out=..., min_vram_gb=..., gguf_file=None, base_model_id=None)
QUEUE = [
    # --- Familias faltantes en el Atlas (caben en Colab Free T4 16 GB) ---
    {"repo": "unsloth/Llama-3.2-3B-Instruct", "out": "meta-llama__llama-3.2-3b-instruct.xray.json", "min_vram_gb": 10},
    {"repo": "unsloth/gemma-2-2b-it", "out": "google__gemma-2-2b-it.xray.json", "min_vram_gb": 10},
    {"repo": "microsoft/Phi-4-mini-instruct", "out": "microsoft__phi-4-mini-instruct.xray.json", "min_vram_gb": 12},
    {"repo": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", "out": "deepseek-ai__deepseek-r1-distill-qwen-1.5b.xray.json", "min_vram_gb": 8},
    # --- Escalera GGUF en Llama-3.2-3B (valida si Q2_K rompe safety/sycophancy tambien en Llama) ---
    {
        "repo": "unsloth/Llama-3.2-3B-Instruct-GGUF",
        "gguf_file": "Llama-3.2-3B-Instruct-IQ4_XS.gguf",
        "base_model_id": "unsloth/Llama-3.2-3B-Instruct",
        "out": "meta-llama__llama-3.2-3b-instruct-iq4_xs.gguf.xray.json",
        "min_vram_gb": 10,
    },
    {
        "repo": "unsloth/Llama-3.2-3B-Instruct-GGUF",
        "gguf_file": "Llama-3.2-3B-Instruct-Q2_K.gguf",
        "base_model_id": "unsloth/Llama-3.2-3B-Instruct",
        "out": "meta-llama__llama-3.2-3b-instruct-q2_k.gguf.xray.json",
        "min_vram_gb": 10,
    },
    # --- Solo en L4 (24 GB) o A100 (40/80 GB) ---
    {"repo": "unsloth/mistral-7b-instruct-v0.3", "out": "mistralai__mistral-7b-instruct-v0.3.xray.json", "min_vram_gb": 20},
]
print(f"Cola configurada: {len(QUEUE)} modelos")


In [ ]:
# @title 3. Ejecutar escaneo X-Ray con auto-resume y limpieza de disco
import gc, json, shutil, sys, time, traceback
sys.path.insert(0, str(REPO_DIR))

from transformers import AutoModelForCausalLM, AutoTokenizer
from wsi.gguf_loader import load_gguf_model
from wsi.xray import scan_model_xray, save_xray_json
from wsi.xray_probes import build_frozen_probe_pack

pack = build_frozen_probe_pack(use_hf_datasets=False)
BATCH_SIZE = 4 if vram_gb < 20 else 8
summary_rows = []

for item in QUEUE:
    out_path = OUT_DIR / item["out"]
    label = f"{item['repo']}:{item['gguf_file']}" if item.get("gguf_file") else item["repo"]

    if out_path.exists():
        print(f"[SKIP] Ya existe: {out_path.name}")
        card = json.loads(out_path.read_text(encoding="utf-8"))
        summary_rows.append((out_path.name, card))
        continue

    if vram_gb < item.get("min_vram_gb", 10):
        print(f"[SKIP] {label} requiere >= {item['min_vram_gb']} GB VRAM (tienes {vram_gb:.1f} GB)")
        continue

    t0 = time.perf_counter()
    print(f"\n=== Escaneando {label} (dtype={DTYPE}, bs={BATCH_SIZE}) ===")
    model = None
    try:
        gmeta = None
        if item.get("gguf_file"):
            model, tok, gmeta = load_gguf_model(
                item["repo"],
                item["gguf_file"],
                base_model_id=item.get("base_model_id"),
                dtype=DTYPE,
                device_map="cuda",
            )
        else:
            tok = AutoTokenizer.from_pretrained(item["repo"], trust_remote_code=True)
            if tok.pad_token_id is None and tok.eos_token_id is not None:
                tok.pad_token = tok.eos_token
            model = AutoModelForCausalLM.from_pretrained(
                item["repo"],
                dtype=DTYPE,
                device_map="cuda",
                trust_remote_code=True,
            ).eval()

        card = scan_model_xray(
            model,
            tok,
            repo_id=label,
            probe_pack=pack,
            device="cuda",
            batch_size=BATCH_SIZE,
        )
        if gmeta is not None:
            card.setdefault("scan_meta", {})["gguf"] = gmeta

        tmp_path = out_path.with_suffix(".tmp")
        save_xray_json(card, str(tmp_path))
        os.replace(tmp_path, out_path)
        summary_rows.append((out_path.name, card))
        print(f"[OK] {out_path.name} en {time.perf_counter()-t0:.1f}s")
    except Exception:
        print(f"[ERROR] Fallo {label}:")
        traceback.print_exc()
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()
        # Limpiar caché HF para no llenar disco de Colab
        hf_cache = Path.home() / ".cache" / "huggingface" / "hub"
        if hf_cache.exists():
            shutil.rmtree(hf_cache, ignore_errors=True)

print("\n=== RESUMEN ===")
for fname, c in summary_rows:
    fp = c["anatomy"]["forward_profile"]
    ov = c["pathologies"]["over_refusal"]
    sy = c["pathologies"]["sycophancy"]
    print(
        f"{fname:52s} | nll={fp['cal_nll']:.2f} | fact={c['anatomy']['fact_readout']['base_fact_top1_acc']:.2f} "
        f"| overref={ov['base_overref_rate']:.2f} | tox={ov['base_toxic_ref_rate']:.2f} "
        f"| framed={ov['base_toxic_framed_ref_rate']:.2f} | syc_cave={sy['cave_rate_false_pushback']}"
    )
